# 96. The Resilient Network Design (Robust Optimization)
## Tier 3: The Advanced Algorithm (Grasshopper Optimization Algorithm)

### Key assumptions
- Grasshopper swarms can effectively explore the solution space for facility selection
- Social interaction functions (attraction/repulsion) guide the search process
- Population-based optimization can escape local optima better than greedy methods
- Binary encoding can represent facility selection decisions effectively
- Adaptive parameters improve convergence behavior over time

### Approach (step-by-step)
1. **Initialize grasshopper population** with random binary facility selections
2. **Define social interaction function** modeling attraction and repulsion forces
3. **Implement position update mechanism** with decreasing coefficient c
4. **Evaluate fitness** considering cost, service coverage, and resilience
5. **Iteratively update positions** using grasshopper swarm dynamics
6. **Track convergence** and identify best solutions found

### What to look for in the results
- Convergence behavior showing improvement over iterations
- Population diversity and exploration patterns
- Comparison with sweep algorithm performance
- Solution quality vs computational time trade-offs

### Concrete example (from the source)
Grasshopper Optimization Algorithm with:
- Population size: 15 grasshoppers
- Maximum iterations: 30
- Problem: 8 facilities, 4 demand zones, 3 disruption scenarios
- Objective: Find optimal facility selection minimizing cost while maintaining 80% service coverage

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Set, Dict
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for Grasshopper Optimization
@dataclass
class Facility:
    """Represents a potential facility location"""
    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float
    
    def distance_to(self, other) -> float:
        """Calculate Euclidean distance to another point"""
        if hasattr(other, 'x') and hasattr(other, 'y'):
            return np.sqrt((self.x - other.x)**2 + (self.y - other.y)**2)
        return 0

@dataclass
class DemandZone:
    """Represents a demand zone"""
    id: int
    x: float
    y: float
    demand: float

@dataclass
class Scenario:
    """Represents a disruption scenario"""
    id: int
    probability: float
    facility_failures: List[int]
    route_blocks: Set[Tuple[int, int]]
    demand_multipliers: Dict[int, float]

@dataclass
class Grasshopper:
    """Represents a grasshopper in the optimization swarm"""
    position: np.ndarray  # Binary vector for facility selection
    fitness: float
    velocity: np.ndarray
    
    def __init__(self, n_facilities: int):
        self.position = np.random.random(n_facilities)
        self.fitness = float('inf')
        self.velocity = np.zeros(n_facilities)

@dataclass
class GOAResult:
    """Results from Grasshopper Optimization Algorithm"""
    best_position: np.ndarray
    best_fitness: float
    best_facilities: List[Facility]
    convergence_history: List[float]
    diversity_history: List[float]
    execution_time: float

In [ ]:
# Create the problem instance
def create_goa_problem():
    """Create the same problem instance as previous tiers for fair comparison"""
    
    # Define 8 potential facilities
    facilities = [
        Facility(1, 10, 20, 1000, 200000),   # Location 1
        Facility(2, 30, 15, 1200, 250000),   # Location 2
        Facility(3, 25, 35, 1100, 230000),   # Location 3
        Facility(4, 45, 30, 1300, 280000),   # Location 4
        Facility(5, 15, 50, 900, 180000),    # Location 5
        Facility(6, 40, 10, 1400, 300000),   # Location 6
        Facility(7, 35, 45, 1150, 240000),   # Location 7
        Facility(8, 20, 40, 1050, 210000),    # Location 8
    ]
    
    # Define 4 demand zones
    demand_zones = [
        DemandZone(1, 15, 25, 200),  # Zone 1
        DemandZone(2, 35, 20, 300),  # Zone 2
        DemandZone(3, 25, 40, 250),  # Zone 3
        DemandZone(4, 40, 35, 350),  # Zone 4
    ]
    
    # Define disruption scenarios
    scenarios = [
        Scenario(
            id=1,
            probability=0.6,
            facility_failures=[],
            route_blocks=set(),
            demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}
        ),
        Scenario(
            id=2,
            probability=0.25,
            facility_failures=[2, 6],
            route_blocks={(1, 3), (4, 2)},
            demand_multipliers={1: 1.5, 2: 0.7, 3: 1.3, 4: 0.8}
        ),
        Scenario(
            id=3,
            probability=0.15,
            facility_failures=[5],
            route_blocks={(3, 1), (7, 4)},
            demand_multipliers={1: 0.8, 2: 1.2, 3: 0.9, 4: 1.1}
        )
    ]
    
    return facilities, demand_zones, scenarios

# Create problem instance
facilities, demand_zones, scenarios = create_goa_problem()

print("GOA Problem Setup:")
print(f"- Facilities: {len(facilities)} potential locations")
print(f"- Demand zones: {len(demand_zones)} service areas")
print(f"- Scenarios: {len(scenarios)} disruption scenarios")
print(f"- Objective: Minimize cost while maintaining ≥80% service coverage")

In [ ]:
# Implement the Grasshopper Optimization Algorithm
class GrasshopperNetworkOptimizer:
    """Grasshopper Optimization Algorithm for resilient network design"""
    
    def __init__(self, facilities: List[Facility], demand_zones: List[DemandZone], 
                 scenarios: List[Scenario], pop_size: int = 15, max_iter: int = 30):
        self.facilities = facilities
        self.demand_zones = demand_zones
        self.scenarios = scenarios
        self.n_facilities = len(facilities)
        self.pop_size = pop_size
        self.max_iter = max_iter
        
        # GOA parameters
        self.c_max = 1.0
        self.c_min = 0.00004
        self.f = 0.5  # Intensity of attraction
        self.l = 1.5  # Attractive length scale
        
        # Initialize population
        self.population = [Grasshopper(self.n_facilities) for _ in range(self.pop_size)]
        
        # Track best solution
        self.best_position = None
        self.best_fitness = float('inf')
        
        # History tracking
        self.convergence_history = []
        self.diversity_history = []
        
    def social_interaction(self, distance: float) -> float:
        """Calculate social interaction force between grasshoppers"""
        if distance == 0:
            return 0
        return self.f * np.exp(-distance / self.l) - np.exp(-distance)
    
    def calculate_service_coverage(self, selected_facilities: List[Facility], 
                                  scenario: Scenario) -> float:
        """Calculate service coverage for a given scenario"""
        available_facilities = [f for f in selected_facilities 
                               if f.id not in scenario.facility_failures]
        
        if not available_facilities:
            return 0.0
        
        total_demand = 0
        served_demand = 0
        
        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            total_demand += actual_demand
            
            best_service = 0
            for facility in available_facilities:
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue
                
                distance = facility.distance_to(demand)
                service_factor = 1 / (1 + distance * 0.01)
                service = min(facility.capacity, actual_demand) * service_factor
                best_service = max(best_service, service)
            
            served_demand += best_service
        
        return served_demand / total_demand if total_demand > 0 else 0
    
    def calculate_transport_cost(self, selected_facilities: List[Facility], 
                               scenario: Scenario) -> float:
        """Calculate transport cost for a scenario"""
        available_facilities = [f for f in selected_facilities 
                               if f.id not in scenario.facility_failures]
        
        total_cost = 0
        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            min_cost = float('inf')
            
            for facility in available_facilities:
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue
                
                distance = facility.distance_to(demand)
                cost = distance * 0.1 * actual_demand
                min_cost = min(min_cost, cost)
            
            if min_cost < float('inf'):
                total_cost += min_cost
        
        return total_cost
    
    def evaluate_fitness(self, position: np.ndarray) -> float:
        """Evaluate fitness of a position (facility selection)"""
        # Convert continuous position to binary selection
        threshold = 0.5
        selected_indices = np.where(position > threshold)[0]
        
        # Ensure at least 2 facilities selected
        if len(selected_indices) < 2:
            return float('inf')
        
        selected_facilities = [self.facilities[i] for i in selected_indices]
        
        # Calculate fixed costs
        fixed_cost = sum(f.fixed_cost for f in selected_facilities)
        
        # Evaluate across scenarios
        total_expected_cost = 0
        worst_service = 1.0
        
        for scenario in self.scenarios:
            service_coverage = self.calculate_service_coverage(selected_facilities, scenario)
            transport_cost = self.calculate_transport_cost(selected_facilities, scenario)
            
            total_cost = fixed_cost + transport_cost
            total_expected_cost += scenario.probability * total_cost
            worst_service = min(worst_service, service_coverage)
        
        # Penalty for not meeting service coverage requirement
        penalty = 0
        if worst_service < 0.8:
            penalty = 1000000 * (0.8 - worst_service)  # Large penalty
        
        # Fitness: cost + penalty - reward for service coverage
        fitness = total_expected_cost + penalty - (worst_service * 100000)
        
        return fitness
    
    def calculate_diversity(self) -> float:
        """Calculate population diversity"""
        if len(self.population) < 2:
            return 0
        
        total_distance = 0
        count = 0
        
        for i in range(len(self.population)):
            for j in range(i + 1, len(self.population)):
                distance = np.linalg.norm(self.population[i].position - self.population[j].position)
                total_distance += distance
                count += 1
        
        return total_distance / count if count > 0 else 0
    
    def update_position(self, iteration: int):
        """Update positions of all grasshoppers"""
        # Calculate decreasing coefficient c
        c = self.c_max - iteration * ((self.c_max - self.c_min) / self.max_iter)
        
        for i, grasshopper in enumerate(self.population):
            social_force = np.zeros(self.n_facilities)
            
            # Calculate social forces from all other grasshoppers
            for j, other in enumerate(self.population):
                if i == j:
                    continue
                
                distance = np.linalg.norm(grasshopper.position - other.position)
                if distance > 0:
                    # Calculate unit vector and social force
                    unit_vector = (other.position - grasshopper.position) / distance
                    social_strength = self.social_interaction(distance)
                    social_force += c * ((self.n_facilities / 2) * social_strength * unit_vector)
            
            # Add attraction to best position
            if self.best_position is not None:
                target_force = c * (self.best_position - grasshopper.position)
                social_force += target_force
            
            # Update position with bounds checking
            new_position = grasshopper.position + social_force
            grasshopper.position = np.clip(new_position, 0, 1)
            
            # Ensure at least some facilities are selected (adaptive threshold)
            if np.sum(grasshopper.position > 0.5) < 2:
                # Randomly activate some facilities
                inactive_indices = np.where(grasshopper.position <= 0.5)[0]
                if len(inactive_indices) > 0:
                    activate_idx = np.random.choice(inactive_indices)
                    grasshopper.position[activate_idx] = 0.7
    
    def optimize(self) -> GOAResult:
        """Run the Grasshopper Optimization Algorithm"""
        start_time = time.time()
        
        print("=" * 60)
        print("GRASSHOPPER OPTIMIZATION ALGORITHM")
        print("=" * 60)
        print(f"Population size: {self.pop_size}")
        print(f"Maximum iterations: {self.max_iter}")
        print(f"Problem size: {self.n_facilities} facilities")
        
        # Initialize and evaluate population
        for grasshopper in self.population:
            grasshopper.fitness = self.evaluate_fitness(grasshopper.position)
            
            # Update best solution
            if grasshopper.fitness < self.best_fitness:
                self.best_fitness = grasshopper.fitness
                self.best_position = grasshopper.position.copy()
        
        print(f"\nInitial best fitness: {self.best_fitness:.2f}")
        
        # Main optimization loop
        for iteration in range(self.max_iter):
            # Update positions
            self.update_position(iteration)
            
            # Evaluate new positions
            for grasshopper in self.population:
                grasshopper.fitness = self.evaluate_fitness(grasshopper.position)
                
                # Update best solution
                if grasshopper.fitness < self.best_fitness:
                    self.best_fitness = grasshopper.fitness
                    self.best_position = grasshopper.position.copy()
            
            # Track convergence and diversity
            self.convergence_history.append(self.best_fitness)
            self.diversity_history.append(self.calculate_diversity())
            
            # Progress reporting
            if iteration % 5 == 0:
                print(f"Iteration {iteration:2d}: Best fitness = {self.best_fitness:.2f}, Diversity = {self.diversity_history[-1]:.3f}")
        
        execution_time = time.time() - start_time
        
        # Convert best position to facility list
        threshold = 0.5
        selected_indices = np.where(self.best_position > threshold)[0]
        best_facilities = [self.facilities[i] for i in selected_indices]
        
        return GOAResult(
            best_position=self.best_position,
            best_fitness=self.best_fitness,
            best_facilities=best_facilities,
            convergence_history=self.convergence_history,
            diversity_history=self.diversity_history,
            execution_time=execution_time
        )

In [ ]:
# Run the Grasshopper Optimization Algorithm
def run_goa():
    """Execute GOA and analyze results"""
    
    # Initialize optimizer
    goa = GrasshopperNetworkOptimizer(
        facilities, demand_zones, scenarios, 
        pop_size=15, max_iter=30
    )
    
    # Run optimization
    result = goa.optimize()
    
    # Analyze results
    print("\n" + "=" * 60)
    print("GOA OPTIMIZATION RESULTS")
    print("=" * 60)
    
    print(f"\nSelected Facilities:")
    for facility in result.best_facilities:
        print(f"  Facility {facility.id}: ({facility.x}, {facility.y}) - Cost: ${facility.fixed_cost:,}")
    
    # Calculate detailed performance metrics
    fixed_cost = sum(f.fixed_cost for f in result.best_facilities)
    
    print(f"\nPerformance Analysis:")
    print(f"  Fixed Cost: ${fixed_cost:,.0f}")
    print(f"  Best Fitness: {result.best_fitness:.2f}")
    print(f"  Execution Time: {result.execution_time:.2f} seconds")
    
    # Evaluate across scenarios
    worst_service = 1.0
    total_expected_cost = 0
    
    print(f"\nScenario Performance:")
    for scenario in scenarios:
        service_coverage = goa.calculate_service_coverage(result.best_facilities, scenario)
        transport_cost = goa.calculate_transport_cost(result.best_facilities, scenario)
        total_cost = fixed_cost + transport_cost
        total_expected_cost += scenario.probability * total_cost
        worst_service = min(worst_service, service_coverage)
        
        print(f"  Scenario {scenario.id}: Coverage = {service_coverage:.1%}, Cost = ${total_cost:,.0f}")
    
    print(f"\nSummary:")
    print(f"  Expected Total Cost: ${total_expected_cost:,.0f}")
    print(f"  Worst-Case Coverage: {worst_service:.1%}")
    print(f"  Service Requirement: {'✓' if worst_service >= 0.8 else '✗'}")
    
    return goa, result

# Run GOA
goa, goa_result = run_goa()

In [ ]:
# Create comprehensive GOA visualization
def create_goa_visualization():
    """Create comprehensive visualization of GOA results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Grasshopper Optimization Algorithm for Resilient Network Design', fontsize=16, fontweight='bold')
    
    # Panel 1: Convergence History
    ax1 = axes[0, 0]
    ax1.set_title('Convergence History', fontweight='bold')
    
    iterations = range(len(goa_result.convergence_history))
    ax1.plot(iterations, goa_result.convergence_history, 'b-', linewidth=2, marker='o', markersize=4)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Best Fitness (lower is better)')
    ax1.grid(True, alpha=0.3)
    
    # Add improvement annotations
    if len(goa_result.convergence_history) > 1:
        initial_fitness = goa_result.convergence_history[0]
        final_fitness = goa_result.convergence_history[-1]
        improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
        ax1.annotate(f'Improvement: {improvement:.1f}%', 
                    xy=(len(iterations)//2, (initial_fitness + final_fitness)/2),
                    ha='center', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # Panel 2: Population Diversity
    ax2 = axes[0, 1]
    ax2.set_title('Population Diversity Over Time', fontweight='bold')
    
    iterations = range(len(goa_result.diversity_history))
    ax2.plot(iterations, goa_result.diversity_history, 'g-', linewidth=2, marker='s', markersize=4)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Population Diversity')
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Solution Visualization
    ax3 = axes[1, 0]
    ax3.set_title('GOA Solution - Facility Selection', fontweight='bold')
    
    # Plot all facilities
    for facility in facilities:
        color = 'green' if facility in goa_result.best_facilities else 'lightgray'
        marker = 'o' if facility in goa_result.best_facilities else 'x'
        size = 300 if facility in goa_result.best_facilities else 200
        ax3.scatter(facility.x, facility.y, c=color, marker=marker, s=size, 
                   alpha=0.7, edgecolors='black', linewidth=2)
        ax3.annotate(f'F{facility.id}', (facility.x, facility.y), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
    # Plot demand zones
    for demand in demand_zones:
        ax3.scatter(demand.x, demand.y, c='blue', marker='s', s=200, 
                   alpha=0.7, edgecolors='black', linewidth=2)
        ax3.annotate(f'D{demand.id}', (demand.x, demand.y), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
    ax3.set_xlabel('X Coordinate')
    ax3.set_ylabel('Y Coordinate')
    ax3.grid(True, alpha=0.3)
    ax3.legend(['Selected Facility', 'Rejected Facility', 'Demand Zone'], loc='upper right')
    
    # Panel 4: Position Distribution
    ax4 = axes[1, 1]
    ax4.set_title('Final Population Position Distribution', fontweight='bold')
    
    # Create heatmap of final positions
    position_matrix = np.array([grasshopper.position for grasshopper in goa.population])
    
    im = ax4.imshow(position_matrix, cmap='RdYlBu_r', aspect='auto')
    ax4.set_xlabel('Facility Index')
    ax4.set_ylabel('Grasshopper Index')
    ax4.set_title('Final Position Matrix (Red=Selected, Blue=Rejected)')
    
    # Add facility labels
    ax4.set_xticks(range(len(facilities)))
    ax4.set_xticklabels([f'F{f.id}' for f in facilities])
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax4)
    cbar.set_label('Selection Probability')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualization
fig = create_goa_visualization()

In [ ]:
# Performance comparison with previous tiers
def compare_algorithms():
    """Compare GOA performance with theoretical benchmarks"""
    
    print("\n" + "=" * 60)
    print("ALGORITHM COMPARISON ANALYSIS")
    print("=" * 60)
    
    # GOA results summary
    print(f"\nGrasshopper Optimization Algorithm:")
    print(f"  Selected Facilities: {[f.id for f in goa_result.best_facilities]}")
    print(f"  Best Fitness: {goa_result.best_fitness:.2f}")
    print(f"  Execution Time: {goa_result.execution_time:.2f} seconds")
    print(f"  Convergence Rate: {len(goa_result.convergence_history)} iterations")
    
    # Calculate actual performance metrics
    fixed_cost = sum(f.fixed_cost for f in goa_result.best_facilities)
    worst_service = 1.0
    total_expected_cost = 0
    
    for scenario in scenarios:
        service_coverage = goa.calculate_service_coverage(goa_result.best_facilities, scenario)
        transport_cost = goa.calculate_transport_cost(goa_result.best_facilities, scenario)
        total_cost = fixed_cost + transport_cost
        total_expected_cost += scenario.probability * total_cost
        worst_service = min(worst_service, service_coverage)
    
    print(f"  Expected Total Cost: ${total_expected_cost:,.0f}")
    print(f"  Worst-Case Coverage: {worst_service:.1%}")
    
    # Algorithm characteristics comparison
    print(f"\nAlgorithm Characteristics:")
    print(f"\nTier 1 (Mathematical Optimization):")
    print(f"  ✓ Guaranteed optimality")
    print(f"  ✓ Comprehensive constraint handling")
    print(f"  ✗ Exponential computational complexity")
    print(f"  ✗ Limited to small problem instances")
    
    print(f"\nTier 2 (Sweep Algorithm):")
    print(f"  ✓ O(n²) computational complexity")
    print(f"  ✓ Geographical diversity optimization")
    print(f"  ✓ Transparent selection process")
    print(f"  ✗ Limited to angular-based patterns")
    
    print(f"\nTier 3 (Grasshopper Optimization):")
    print(f"  ✓ Population-based global search")
    print(f"  ✓ Ability to escape local optima")
    print(f"  ✓ Adaptive parameter control")
    print(f"  ✓ Balance between exploration and exploitation")
    print(f"  ⚠ No optimality guarantee")
    print(f"  ⚠ Parameter sensitivity")
    
    # Convergence analysis
    if len(goa_result.convergence_history) > 1:
        print(f"\nConvergence Analysis:")
        initial_fitness = goa_result.convergence_history[0]
        final_fitness = goa_result.convergence_history[-1]
        improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
        
        print(f"  Initial fitness: {initial_fitness:.2f}")
        print(f"  Final fitness: {final_fitness:.2f}")
        print(f"  Total improvement: {improvement:.1f}%")
        
        # Find iteration with most improvement
        best_improvement = 0
        best_iteration = 0
        for i in range(1, len(goa_result.convergence_history)):
            iter_improvement = goa_result.convergence_history[i-1] - goa_result.convergence_history[i]
            if iter_improvement > best_improvement:
                best_improvement = iter_improvement
                best_iteration = i
        
        print(f"  Best single improvement: {best_improvement:.2f} at iteration {best_iteration}")
    
    # Diversity analysis
    if goa_result.diversity_history:
        initial_diversity = goa_result.diversity_history[0]
        final_diversity = goa_result.diversity_history[-1]
        diversity_change = ((final_diversity - initial_diversity) / initial_diversity) * 100 if initial_diversity > 0 else 0
        
        print(f"\nDiversity Analysis:")
        print(f"  Initial diversity: {initial_diversity:.3f}")
        print(f"  Final diversity: {final_diversity:.3f}")
        print(f"  Diversity change: {diversity_change:+.1f}%")
        print(f"  Convergence behavior: {'Good' if abs(diversity_change) < 50 else 'Poor'}")
    
    return goa_result

# Compare algorithms
comparison_result = compare_algorithms()

In [ ]:
# Parameter sensitivity analysis
def parameter_sensitivity_analysis():
    """Analyze sensitivity of GOA to different parameters"""
    
    print("\n" + "=" * 60)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 60)
    
    # Test different population sizes
    pop_sizes = [5, 10, 15, 20]
    results = []
    
    print(f"\nTesting different population sizes:")
    for pop_size in pop_sizes:
        goa_test = GrasshopperNetworkOptimizer(
            facilities, demand_zones, scenarios, 
            pop_size=pop_size, max_iter=20  # Reduced iterations for speed
        )
        
        start_time = time.time()
        result = goa_test.optimize()
        execution_time = time.time() - start_time
        
        # Calculate actual performance
        fixed_cost = sum(f.fixed_cost for f in result.best_facilities)
        worst_service = 1.0
        total_expected_cost = 0
        
        for scenario in scenarios:
            service_coverage = goa_test.calculate_service_coverage(result.best_facilities, scenario)
            transport_cost = goa_test.calculate_transport_cost(result.best_facilities, scenario)
            total_cost = fixed_cost + transport_cost
            total_expected_cost += scenario.probability * total_cost
            worst_service = min(worst_service, service_coverage)
        
        results.append({
            'pop_size': pop_size,
            'fitness': result.best_fitness,
            'cost': total_expected_cost,
            'coverage': worst_service,
            'time': execution_time,
            'facilities': len(result.best_facilities)
        })
        
        print(f"  Pop Size {pop_size:2d}: Fitness={result.best_fitness:8.2f}, "
              f"Cost=${total_expected_cost:8.0f}, Coverage={worst_service:5.1%}, "
              f"Time={execution_time:5.2f}s")
    
    # Create sensitivity visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('GOA Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    pop_size_vals = [r['pop_size'] for r in results]
    
    # Fitness vs Population Size
    axes[0, 0].plot(pop_size_vals, [r['fitness'] for r in results], 'bo-', linewidth=2, markersize=8)
    axes[0, 0].set_xlabel('Population Size')
    axes[0, 0].set_ylabel('Best Fitness')
    axes[0, 0].set_title('Fitness vs Population Size')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Cost vs Population Size
    axes[0, 1].plot(pop_size_vals, [r['cost'] for r in results], 'go-', linewidth=2, markersize=8)
    axes[0, 1].set_xlabel('Population Size')
    axes[0, 1].set_ylabel('Expected Total Cost ($)')
    axes[0, 1].set_title('Cost vs Population Size')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Coverage vs Population Size
    axes[1, 0].plot(pop_size_vals, [r['coverage'] * 100 for r in results], 'mo-', linewidth=2, markersize=8)
    axes[1, 0].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% Requirement')
    axes[1, 0].set_xlabel('Population Size')
    axes[1, 0].set_ylabel('Worst-Case Coverage (%)')
    axes[1, 0].set_title('Coverage vs Population Size')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # Time vs Population Size
    axes[1, 1].plot(pop_size_vals, [r['time'] for r in results], 'co-', linewidth=2, markersize=8)
    axes[1, 1].set_xlabel('Population Size')
    axes[1, 1].set_ylabel('Execution Time (seconds)')
    axes[1, 1].set_title('Computational Time vs Population Size')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Run sensitivity analysis
sensitivity_results = parameter_sensitivity_analysis()

### Why this Tier exists vs earlier Tiers
This Tier 3 provides advanced metaheuristic optimization when simpler methods are insufficient:

**Key Advantages over Tier 1 & 2:**
- **Global search capability**: Can escape local optima that trap greedy methods
- **Population-based exploration**: Multiple solutions explored simultaneously
- **Adaptive behavior**: Dynamic parameter adjustment improves convergence
- **Balance of exploration/exploitation**: Systematic trade-off between search breadth and depth
- **Scalability to complex landscapes**: Handles non-convex optimization problems

**When to prefer this Tier:**
- Complex solution spaces with many local optima
- Problems where traditional heuristics get stuck in suboptimal solutions
- Situations requiring balance between exploration and exploitation
- Medium to large-scale problems where exact methods are too slow
- Multi-modal optimization landscapes with diverse solution regions

### Pros / Cons vs earlier Tiers
**Pros:**
- Superior ability to find global optima in complex landscapes
- Less likely to get trapped in local optima than greedy methods
- Population-based approach provides robustness to initial conditions
- Adaptive parameters improve convergence over time
- Good balance between computational efficiency and solution quality

**Cons:**
- No guarantee of finding the true global optimum
- Parameter tuning required for best performance
- Higher computational cost than simple heuristics
- Convergence can be sensitive to parameter settings
- May require multiple runs for consistent results

### When to use this Tier
- **Complex facility location problems** with non-linear cost structures
- **Multi-objective optimization** where trade-offs are important
- **Large-scale networks** where sweep algorithms may be too restrictive
- **Dynamic environments** requiring adaptive search strategies
- **Research and development** exploring new solution approaches
- **Benchmark studies** comparing different optimization methodologies